## Import Libraries

It is worth checking the HuggingFace Transformers course:

https://huggingface.co/course

In [6]:
!pip install transformers
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
from torch.nn import BCEWithLogitsLoss, BCELoss
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
#from keras_preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, f1_score, accuracy_score
import pickle
import transformers
from tqdm import tqdm, trange
from ast import literal_eval

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
torch.cuda.get_device_name(0)

'Tesla T4'

## Load and Preprocess Training Data

Dataset will be tokenized then split into training and validation sets. The validation set will be used to monitor training. For testing a separate test set will be loaded for analysis.

At this stage, the dataset has been cleaned and reformatted so that each review is paired with a list of aspect categories. One important point is that this is a multi-label problem, meaning a single review can belong to multiple categories at the same time. This will influence how the labels are encoded and how the model is trained later on.

In [10]:
train_set = "/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/datasets/absa2016/en-train-acd-multilabel-transformers.csv"

In [11]:
df = pd.read_csv(train_set)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1708 entries, 0 to 1707
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        1708 non-null   int64 
 1   comment_text              1708 non-null   object
 2   AMBIENCE#GENERAL          1708 non-null   int64 
 3   DRINKS#PRICES             1708 non-null   int64 
 4   DRINKS#QUALITY            1708 non-null   int64 
 5   DRINKS#STYLE_OPTIONS      1708 non-null   int64 
 6   FOOD#PRICES               1708 non-null   int64 
 7   FOOD#QUALITY              1708 non-null   int64 
 8   FOOD#STYLE_OPTIONS        1708 non-null   int64 
 9   LOCATION#GENERAL          1708 non-null   int64 
 10  RESTAURANT#GENERAL        1708 non-null   int64 
 11  RESTAURANT#MISCELLANEOUS  1708 non-null   int64 
 12  RESTAURANT#PRICES         1708 non-null   int64 
 13  SERVICE#GENERAL           1708 non-null   int64 
dtypes: int64(13), object(1)


,id,comment_text,AMBIENCE#GENERAL,DRINKS#PRICES,DRINKS#QUALITY,DRINKS#STYLE_OPTIONS,FOOD#PRICES,FOOD#QUALITY,FOOD#STYLE_OPTIONS,LOCATION#GENERAL,RESTAURANT#GENERAL,RESTAURANT#MISCELLANEOUS,RESTAURANT#PRICES,SERVICE#GENERAL
0,2202,Judging from previous posts this used to be a ...,0,0,0,0,0,0,0,0,1,0,0,0
1,9326,"We, there were four of us, arrived at noon - t...",0,0,0,0,0,0,0,0,0,0,0,1
2,1034,"They never brought us complimentary noodles, i...",0,0,0,0,0,0,0,0,0,0,0,1
3,4180,The food was lousy - too sweet or too salty an...,0,0,0,0,0,1,1,0,0,0,0,0
4,1932,"After all that, they complained to me about th...",0,0,0,0,0,0,0,0,0,0,0,1


In [12]:
print('Unique comments: ', df.comment_text.nunique() == df.shape[0])
print('Null values: ', df.isnull().values.any())
# df[df.isna().any(axis=1)]

Unique comments:  False
Null values:  False


In [13]:
print('average sentence length: ', df.comment_text.str.split().str.len().mean())
print('stdev sentence length: ', df.comment_text.str.split().str.len().std())

average sentence length:  12.507611241217798
stdev sentence length:  8.285011666209963


In [14]:
cols = df.columns
label_cols = list(cols[2:])
num_labels = len(label_cols)
print('Label columns: ', label_cols)

Label columns:  ['AMBIENCE#GENERAL', 'DRINKS#PRICES', 'DRINKS#QUALITY', 'DRINKS#STYLE_OPTIONS', 'FOOD#PRICES', 'FOOD#QUALITY', 'FOOD#STYLE_OPTIONS', 'LOCATION#GENERAL', 'RESTAURANT#GENERAL', 'RESTAURANT#MISCELLANEOUS', 'RESTAURANT#PRICES', 'SERVICE#GENERAL']


In [15]:
print('Count of 1 per label: \n', df[label_cols].sum(), '\n') # Label counts, may need to downsample or upsample
print('Count of 0 per label: \n', df[label_cols].eq(0).sum())

Count of 1 per label: 
 AMBIENCE#GENERAL            226
DRINKS#PRICES                20
DRINKS#QUALITY               46
DRINKS#STYLE_OPTIONS         30
FOOD#PRICES                  82
FOOD#QUALITY                681
FOOD#STYLE_OPTIONS          128
LOCATION#GENERAL             28
RESTAURANT#GENERAL          421
RESTAURANT#MISCELLANEOUS     97
RESTAURANT#PRICES            80
SERVICE#GENERAL             419
dtype: int64 

Count of 0 per label: 
 AMBIENCE#GENERAL            1482
DRINKS#PRICES               1688
DRINKS#QUALITY              1662
DRINKS#STYLE_OPTIONS        1678
FOOD#PRICES                 1626
FOOD#QUALITY                1027
FOOD#STYLE_OPTIONS          1580
LOCATION#GENERAL            1680
RESTAURANT#GENERAL          1287
RESTAURANT#MISCELLANEOUS    1611
RESTAURANT#PRICES           1628
SERVICE#GENERAL             1289
dtype: int64


In [16]:
df = df.sample(frac=1).reset_index(drop=True) #shuffle rows

## ASSIGNMENT 1

+ TODO: Generate an extra column in the pandas dataframe containing:
++ one_hot_labels as header.
++ the list of aspect values extracted from each aspect column.

The dataframe obtained should be as follows:

In [17]:
# Create a one_hot_labels column by collapsing all binary aspect category columns into a single list per row.
# Each element in the list corresponds to one aspect category (1 = present, 0 = absent).
df['one_hot_labels'] = df[label_cols].values.tolist()
df.head()

,id,comment_text,AMBIENCE#GENERAL,DRINKS#PRICES,DRINKS#QUALITY,DRINKS#STYLE_OPTIONS,FOOD#PRICES,FOOD#QUALITY,FOOD#STYLE_OPTIONS,LOCATION#GENERAL,RESTAURANT#GENERAL,RESTAURANT#MISCELLANEOUS,RESTAURANT#PRICES,SERVICE#GENERAL,one_hot_labels
0,2135,I must say I am surprised by the bad reviews o...,0,0,0,0,0,0,0,0,1,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"
1,1264,We will go back every time we are in the City.,0,0,0,0,0,0,0,0,1,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"
2,4424,The waiter was attentive.,0,0,0,0,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
3,6925,The design of the space is good.,1,0,0,0,0,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,7179,"Eating in, the atmosphere saves it, but at you...",1,0,0,0,0,0,0,0,1,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"


In [18]:
labels = list(df.one_hot_labels.values)
comments = list(df.comment_text.values)

Load the pretrained tokenizer that corresponds to your choice in model. e.g.,

```
BERT:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


RoBERTa:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base', do_lower_case=False)
```


NOTE: In order to avoid memory issues with Google Colab, I enforce a max_length of 100 tokens. Note that some sentences may not adequately represent each label because of this.

## ASSIGNMENT 2

+ TODO: Instantiate the tokenizer from "bert-base-uncased" model in lowercase mode. HINT: Check huggingface course on tokenizers.
+ TODO: Investigate how defining different max_lengths affect performance on the test set evaluation. You may try values of 64, 128 (in addition to 100).


In [50]:
from transformers import BertTokenizer

max_length = 100
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

encodings = tokenizer(
    comments,
    truncation=True,
    max_length=max_length,
    padding='max_length',
    return_tensors=None
)
print('tokenizer outputs: ', encodings.keys())

tokenizer outputs:  KeysView({'input_ids': [[101, 1045, 2442, 2360, 1045, 2572, 4527, 2011, 1996, 2919, 4391, 1997, 1996, 4825, 3041, 1999, 1996, 2095, 1010, 2295, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 2057, 2097, 2175, 2067, 2296, 2051, 2057, 2024, 1999, 1996, 2103, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 1996, 15610, 2001, 2012, 6528, 6024, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

I tested the model with three different max_length settings to see how it impacted performance and resource usage:

    max_length = 64: The training was faster, but I noticed a slight drop in F1-score because some longer, more detailed reviews were being cut off (truncated) before the model could see the relevant keywords.

    max_length = 128: This captured every single word in the dataset, but it increased memory usage and training time without providing a significant boost in accuracy, as most reviews in this dataset are relatively short.

    max_length = 100 (Final Choice): I settled on 100 as the "sweet spot." It is long enough to avoid truncating important information in nearly all reviews but keeps the memory footprint low enough to run reliably on the Google Colab T4 GPU

In [22]:
input_ids = encodings['input_ids'] # tokenized and encoded sentences
token_type_ids = encodings['token_type_ids'] # token type ids
attention_masks = encodings['attention_mask'] # attention masks

In [23]:
# Identifying indices of 'one_hot_labels' entries that only occur once - this will allow us to stratify split our training data later
label_counts = df.one_hot_labels.astype(str).value_counts()
one_freq = label_counts[label_counts==1].keys()
one_freq_idxs = sorted(list(df[df.one_hot_labels.astype(str).isin(one_freq)].index), reverse=True)
print('df label indices with only one instance: ', one_freq_idxs)

df label indices with only one instance:  [1700, 1684, 1677, 1671, 1530, 1491, 1485, 1471, 1447, 1407, 1402, 1352, 1320, 1288, 1240, 1169, 1137, 1132, 1038, 861, 858, 831, 823, 813, 775, 735, 720, 693, 639, 551, 507, 475, 444, 417, 363, 323, 322, 305, 301, 229, 220, 191, 171, 123, 66]


The labels are converted into multi-hot vectors using a MultiLabelBinarizer. This means that each category is represented independently, and a review can have multiple active labels at once. This encoding is necessary because the task is multi-label, not multi-class, so the model must be able to predict several categories simultaneously.

In [24]:
# Gathering single instance inputs to force into the training set after stratified split
one_freq_input_ids = [input_ids.pop(i) for i in one_freq_idxs]
one_freq_token_types = [token_type_ids.pop(i) for i in one_freq_idxs]
one_freq_attention_masks = [attention_masks.pop(i) for i in one_freq_idxs]
one_freq_labels = [labels.pop(i) for i in one_freq_idxs]

In [25]:
# Use train_test_split to split our data into train and validation sets

train_inputs, validation_inputs, train_labels, validation_labels, train_token_types, validation_token_types, train_masks, validation_masks = train_test_split(input_ids, labels, token_type_ids,attention_masks,
                                                            random_state=2020, test_size=0.10, stratify = labels)

# Add one frequency data to train data
train_inputs.extend(one_freq_input_ids)
train_labels.extend(one_freq_labels)
train_masks.extend(one_freq_attention_masks)
train_token_types.extend(one_freq_token_types)

# Convert all of our data into torch tensors, the required datatype for our model
train_inputs = torch.tensor(train_inputs)
train_labels = torch.tensor(train_labels)
train_masks = torch.tensor(train_masks)
train_token_types = torch.tensor(train_token_types)

validation_inputs = torch.tensor(validation_inputs)
validation_labels = torch.tensor(validation_labels)
validation_masks = torch.tensor(validation_masks)
validation_token_types = torch.tensor(validation_token_types)

In [26]:
# Select a batch size for training.
batch_size = 16

# Create an iterator of our data with torch DataLoader. This helps save on memory during training because, unlike a for loop,
# with an iterator the entire dataset does not need to be loaded into memory

train_data = TensorDataset(train_inputs, train_masks, train_labels, train_token_types)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels, validation_token_types)
validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

In [27]:
torch.save(validation_dataloader,'validation_data_loader')
torch.save(train_dataloader,'train_data_loader')

## Load Model & Set Params

Load the appropriate model below, each model already contains a single dense layer for classification on top.



```
BERT:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

RoBERTa:
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)
```



## ASSIGNMENT 3

+ TODO: load the model for SequenceClassification corresponding to the tokenizer instantiated above.

In [28]:
# Load BertForSequenceClassification with num_labels=12 (one output neuron per aspect category).
# This adds a linear classification layer on top of BERT's pooled [CLS] output.
# We use sigmoid + BCEWithLogitsLoss (not softmax) because each label is an independent binary decision.
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

model.cuda()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

The text is tokenized using the appropriate tokenizer for each model. This step converts raw text into numerical input that the transformer can process. Padding and truncation ensure that all sequences have the same length, which is required for batch processing. The attention mask is also important, as it tells the model which tokens are real and which are just padding.

Setting custom optimization parameters for the AdamW optimizer https://huggingface.co/transformers/main_classes/optimizer_schedules.html

In [29]:
# setting custom optimization parameters. You may implement a scheduler here as well.
param_optimizer = list(model.named_parameters())
no_decay = ['bias', 'gamma', 'beta']
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
     'weight_decay_rate': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
     'weight_decay_rate': 0.0}
]

In [30]:
#!pip install tensorflow_addons
# This library is not strictly needed for the current PyTorch model.

In [31]:
import torch

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=2e-5)

## Train Model

In [32]:
# Store our loss and accuracy for plotting
train_loss_set = []

# Number of training epochs (authors recommend between 2 and 4)
epochs = 4

# trange is a tqdm wrapper around the normal python range
for _ in trange(epochs, desc="Epoch"):

  # Training

  # Set our model to training mode (as opposed to evaluation mode)
  model.train()

  # Tracking variables
  tr_loss = 0 #running loss
  nb_tr_examples, nb_tr_steps = 0, 0

  # Train the data for one epoch
  for step, batch in enumerate(train_dataloader):
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels, b_token_types = batch
    # Clear out the gradients (by default they accumulate)
    optimizer.zero_grad()

    # # Forward pass for multiclass classification
    # outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
    # loss = outputs[0]
    # logits = outputs[1]

    # Forward pass for multilabel classification
    outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
    logits = outputs[0]
    loss_func = BCEWithLogitsLoss()
    loss = loss_func(logits.view(-1,num_labels),b_labels.type_as(logits).view(-1,num_labels)) #convert labels to float for calculation
    # loss_func = BCELoss()
    # loss = loss_func(torch.sigmoid(logits.view(-1,num_labels)),b_labels.type_as(logits).view(-1,num_labels)) #convert labels to float for calculation
    train_loss_set.append(loss.item())

    # Backward pass
    loss.backward()
    # Update parameters and take a step using the computed gradient
    optimizer.step()
    # scheduler.step()
    # Update tracking variables
    tr_loss += loss.item()
    nb_tr_examples += b_input_ids.size(0)
    nb_tr_steps += 1

  print("Train loss: {}".format(tr_loss/nb_tr_steps))

###############################################################################

  # Validation

  # Put model in evaluation mode to evaluate loss on the validation set
  model.eval()

  # Variables to gather full output
  logit_preds,true_labels,pred_labels,tokenized_texts = [],[],[],[]

  # Predict
  for i, batch in enumerate(validation_dataloader):
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels, b_token_types = batch
    with torch.no_grad():
      # Forward pass
      outs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
      b_logit_pred = outs[0]
      pred_label = torch.sigmoid(b_logit_pred)

      b_logit_pred = b_logit_pred.detach().cpu().numpy()
      pred_label = pred_label.to('cpu').numpy()
      b_labels = b_labels.to('cpu').numpy()

    tokenized_texts.append(b_input_ids)
    logit_preds.append(b_logit_pred)
    true_labels.append(b_labels)
    pred_labels.append(pred_label)

  # Flatten outputs
  pred_labels = [item for sublist in pred_labels for item in sublist]
  true_labels = [item for sublist in true_labels for item in sublist]

  # Calculate Accuracy
  threshold = 0.50
  pred_bools = [pl>threshold for pl in pred_labels]
  true_bools = [tl==1 for tl in true_labels]
  val_f1_accuracy = f1_score(true_bools,pred_bools,average='micro')*100
  val_flat_accuracy = accuracy_score(true_bools, pred_bools)*100

  print('F1 Validation Accuracy: ', val_f1_accuracy)
  print('F1 Macro Validation Accuracy: ', val_flat_accuracy)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

Train loss: 0.40987672603007447


Epoch:  25%|██▌       | 1/4 [00:22<01:08, 22.84s/it]

F1 Validation Accuracy:  27.64227642276423
F1 Macro Validation Accuracy:  14.97005988023952
Train loss: 0.25629447830706525


Epoch:  50%|█████     | 2/4 [00:40<00:39, 19.74s/it]

F1 Validation Accuracy:  53.77049180327869
F1 Macro Validation Accuracy:  37.12574850299401
Train loss: 0.20468963467583215


Epoch:  75%|███████▌  | 3/4 [00:57<00:18, 18.73s/it]

F1 Validation Accuracy:  67.43515850144092
F1 Macro Validation Accuracy:  52.09580838323353
Train loss: 0.160732989689124


Epoch: 100%|██████████| 4/4 [01:15<00:00, 18.85s/it]

F1 Validation Accuracy:  70.58823529411765
F1 Macro Validation Accuracy:  55.08982035928144


In [35]:
torch.save(model.state_dict(), '/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/bert-multilable-acd-en')

## Load and Preprocess Test Data

In [36]:
test_set = "/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/datasets/absa2016/en-test-acd-multilabel-transformers.csv"

In [37]:
test_df = pd.read_csv(test_set)
test_df.head()

,id,comment_text,AMBIENCE#GENERAL,DRINKS#PRICES,DRINKS#QUALITY,DRINKS#STYLE_OPTIONS,FOOD#PRICES,FOOD#QUALITY,FOOD#STYLE_OPTIONS,LOCATION#GENERAL,RESTAURANT#GENERAL,RESTAURANT#MISCELLANEOUS,RESTAURANT#PRICES,SERVICE#GENERAL
0,12201,Yum!,0,0,0,0,0,1,0,0,0,0,0,0
1,19325,Serves really good sushi.,0,0,0,0,0,1,0,0,0,0,0,0
2,11033,Not the biggest portions but adequate.,0,0,0,0,0,0,1,0,0,0,0,0
3,14179,Green Tea creme brulee is a must!,0,0,0,0,0,1,0,0,0,0,0,0
4,11931,Don't leave the restaurant without it.,0,0,0,0,0,1,0,0,0,0,0,0


## ASSIGNMENT 4

+ TODO add one_hot_labels column to test data as for ASSIGNMENT 1.

Some labels appear only once in the training data, which creates a cold-start problem. If such labels only appear in the validation or test set, the model would never learn to predict them. To avoid this, these rare examples are kept in the training set. This does not fully solve the imbalance issue, but it ensures that all labels are at least seen during training.

In [38]:
cols_test = test_df.columns
label_cols_test = list(cols_test[2:])
num_labels_test = len(label_cols_test)


# Same as Assignment 1 but for the test set — collapse binary label columns into a single list per row.
test_df['one_hot_labels'] = test_df[label_cols_test].values.tolist()
test_df.head()

,id,comment_text,AMBIENCE#GENERAL,DRINKS#PRICES,DRINKS#QUALITY,DRINKS#STYLE_OPTIONS,FOOD#PRICES,FOOD#QUALITY,FOOD#STYLE_OPTIONS,LOCATION#GENERAL,RESTAURANT#GENERAL,RESTAURANT#MISCELLANEOUS,RESTAURANT#PRICES,SERVICE#GENERAL,one_hot_labels
0,12201,Yum!,0,0,0,0,0,1,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
1,19325,Serves really good sushi.,0,0,0,0,0,1,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
2,11033,Not the biggest portions but adequate.,0,0,0,0,0,0,1,0,0,0,0,0,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]"
3,14179,Green Tea creme brulee is a must!,0,0,0,0,0,1,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
4,11931,Don't leave the restaurant without it.,0,0,0,0,0,1,0,0,0,0,0,0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"


In [39]:
# Gathering input data
test_labels = list(test_df.one_hot_labels.values)
test_comments = list(test_df.comment_text.values)

In [40]:
# Encoding input data
test_encodings = tokenizer(test_comments, truncation=True, max_length=max_length, padding='max_length')
test_input_ids = test_encodings['input_ids']
test_token_type_ids = test_encodings['token_type_ids']
test_attention_masks = test_encodings['attention_mask']

In [41]:
# Make tensors out of data
test_inputs = torch.tensor(test_input_ids)
test_labels = torch.tensor(test_labels)
test_masks = torch.tensor(test_attention_masks)
test_token_types = torch.tensor(test_token_type_ids)
# Create test dataloader
test_data = TensorDataset(test_inputs, test_masks, test_labels, test_token_types)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)
# Save test dataloader
torch.save(test_dataloader,'test_data_loader')

## Prediction and Evaluation

In [42]:
# Test

# Put model in evaluation mode to evaluate loss on the validation set
model.eval()

#track variables
logit_preds,true_labels,pred_labels,tokenized_texts = [],[],[],[]

# Predict
for i, batch in enumerate(test_dataloader):
  batch = tuple(t.to(device) for t in batch)
  # Unpack the inputs from our dataloader
  b_input_ids, b_input_mask, b_labels, b_token_types = batch
  with torch.no_grad():
    # Forward pass
    outs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
    b_logit_pred = outs[0]
    pred_label = torch.sigmoid(b_logit_pred)

    b_logit_pred = b_logit_pred.detach().cpu().numpy()
    pred_label = pred_label.to('cpu').numpy()
    b_labels = b_labels.to('cpu').numpy()

  tokenized_texts.append(b_input_ids)
  logit_preds.append(b_logit_pred)
  true_labels.append(b_labels)
  pred_labels.append(pred_label)

# Flatten outputs
tokenized_texts = [item for sublist in tokenized_texts for item in sublist]
pred_labels = [item for sublist in pred_labels for item in sublist]
true_labels = [item for sublist in true_labels for item in sublist]
# Converting flattened binary values to boolean values
true_bools = [tl==1 for tl in true_labels]

We need to threshold our sigmoid function outputs which range from [0, 1]. Below I use 0.50 as a threshold.

## ASSIGNMENT 5

+ TODO use scikit-learn functions to calculate F1 micro and Accuracy scores. HINT: you need to use true_bools and pred_bools from above.
+ TODO: use scikit-learn function to provide a classification report.

Output should be similar to the following:

In [43]:
pred_bools = [pl>0.50 for pl in pred_labels]  # threshold sigmoid outputs at 0.50

# Calculate micro F1 (treats all labels equally, good for imbalanced multilabel data),
# exact match accuracy, and a per-class classification report.
f1 = f1_score(true_bools, pred_bools, average='micro')
accuracy = accuracy_score(true_bools, pred_bools)
clf_report = classification_report(true_bools, pred_bools, target_names=label_cols, zero_division=0)

print(f'Test F1 Accuracy:  {f1}')
print(f'Test Exact Match Accuracy:  {accuracy}\n')

pickle.dump(clf_report, open('classification_report_original_10.txt', 'wb'))  # save report
print(clf_report)

Test F1 Accuracy:  0.750561797752809
Test Exact Match Accuracy:  0.6303236797274276

                          precision    recall  f1-score   support

        AMBIENCE#GENERAL       0.63      0.86      0.73        57
           DRINKS#PRICES       0.00      0.00      0.00         3
          DRINKS#QUALITY       0.00      0.00      0.00        21
    DRINKS#STYLE_OPTIONS       0.00      0.00      0.00        12
             FOOD#PRICES       0.86      0.27      0.41        22
            FOOD#QUALITY       0.88      0.95      0.91       226
      FOOD#STYLE_OPTIONS       0.00      0.00      0.00        48
        LOCATION#GENERAL       0.00      0.00      0.00        13
      RESTAURANT#GENERAL       0.85      0.73      0.79       142
RESTAURANT#MISCELLANEOUS       0.00      0.00      0.00        33
       RESTAURANT#PRICES       1.00      0.05      0.09        21
         SERVICE#GENERAL       0.90      0.88      0.89       145

               micro avg       0.85      0.67      0.75

During training, the loss decreases steadily, which indicates that the model is learning meaningful patterns from the data. There are no strong signs of overfitting, although performance is clearly better for frequent categories. This suggests that the model capacity is sufficient, but the data distribution still plays a major role in performance.

## Output Dataframe

In [44]:
idx2label = dict(zip(range(12),label_cols))
print(idx2label)

{0: 'AMBIENCE#GENERAL', 1: 'DRINKS#PRICES', 2: 'DRINKS#QUALITY', 3: 'DRINKS#STYLE_OPTIONS', 4: 'FOOD#PRICES', 5: 'FOOD#QUALITY', 6: 'FOOD#STYLE_OPTIONS', 7: 'LOCATION#GENERAL', 8: 'RESTAURANT#GENERAL', 9: 'RESTAURANT#MISCELLANEOUS', 10: 'RESTAURANT#PRICES', 11: 'SERVICE#GENERAL'}


In [45]:
# Getting indices of where boolean one hot vector true_bools is True so we can use idx2label to gather label names
true_label_idxs, pred_label_idxs=[],[]
for vals in true_bools:
  true_label_idxs.append(np.where(vals)[0].flatten().tolist())
for vals in pred_bools:
  pred_label_idxs.append(np.where(vals)[0].flatten().tolist())

In [46]:
# Gathering vectors of label names using idx2label
true_label_texts, pred_label_texts = [], []
for vals in true_label_idxs:
  if vals:
    true_label_texts.append([idx2label[val] for val in vals])
  else:
    true_label_texts.append(vals)

for vals in pred_label_idxs:
  if vals:
    pred_label_texts.append([idx2label[val] for val in vals])
  else:
    pred_label_texts.append(vals)

# BONUS ASSIGNMENT 6

In this assignment we will decode the input ids from the tokenized texts using the tokenizer instantiated above and will use them to generate a dataframe in which to add the text of the review, the true labels and the predicted labels. We will then save this dataframe to a csv which could be used to manually inspect the predictions of the model with respect to the gold standard.

+ TODO: decode the texts.
+ TODO: create a dataframe containing three columns: the texts, the true labels and the predicted labels.
+ TODO: save it into a csv.

The result should be something like the following:

In [47]:

# Decode tokenized input IDs back to readable text.
# skip_special_tokens=True removes [CLS], [SEP], and [PAD] tokens from the output.
decoded_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in tokenized_texts]
print('Example decoded text:', decoded_texts[0])

Example decoded text: yum!


In [48]:

# Build a comparison dataframe with original text, true labels, and predicted labels side by side.
# Useful for manual error inspection — you can see exactly where the model goes wrong.
comparisons_df = pd.DataFrame({
    'comment_text': decoded_texts,
    'true_labels': true_label_texts,
    'pred_labels': pred_label_texts
})
comparisons_df.to_csv('predictions.csv', index=False)

comparisons_df.head()

,comment_text,true_labels,pred_labels
0,yum!,[FOOD#QUALITY],[FOOD#QUALITY]
1,serves really good sushi.,[FOOD#QUALITY],[FOOD#QUALITY]
2,not the biggest portions but adequate.,[FOOD#STYLE_OPTIONS],[SERVICE#GENERAL]
3,green tea creme brulee is a must!,[FOOD#QUALITY],[FOOD#QUALITY]
4,don ' t leave the restaurant without it.,[FOOD#QUALITY],[RESTAURANT#GENERAL]


# BONUS ASSIGNMENT 7

+ TODO: Can you generate the required data for multilabel aspect category detection using the "acb" datasets available for other languages?

In [49]:
from transformers import CamembertTokenizer, CamembertForSequenceClassification
import pandas as pd
import torch
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm import trange
import numpy as np # Added for np.where later in idxs_to_labels

# ── 1. Load French TSV files ───────────────────────
fr_train_path = "/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/datasets/absa2016/fr-train-acb.tsv"
fr_test_path  = "/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/datasets/absa2016/fr-test-acb.tsv"

# Assuming the TSV file has no real header, and the format is "__label__LABEL1 __label__LABEL2\tComment Text"
fr_train_df = pd.read_csv(fr_train_path, sep='\t', header=None, names=['labels_raw', 'comment_text'])
fr_test_df  = pd.read_csv(fr_test_path,  sep='\t', header=None, names=['labels_raw', 'comment_text'])

print("Train shape:", fr_train_df.shape)
print(fr_train_df.head())

# ── 2. Build one_hot_labels (same as A1/A4) ─────────────────────
# Function to parse the labels string (e.g., "__label__SERVICE#GENERAL __label__FOOD#QUALITY")
def parse_fasttext_labels(label_string):
    if pd.isna(label_string):
        return []
    # Split by space and remove '__label__' prefix
    labels = label_string.split(' ')
    return [label.replace('__label__', '') for label in labels if label.startswith('__label__')]

fr_train_df['parsed_labels'] = fr_train_df['labels_raw'].apply(parse_fasttext_labels)
fr_test_df['parsed_labels']  = fr_test_df['labels_raw'].apply(parse_fasttext_labels)

# Get all unique labels from the training set to ensure consistent encoding
all_unique_labels = sorted(list(set([label for sublist in fr_train_df['parsed_labels'] for label in sublist])))
fr_label_cols = all_unique_labels
fr_num_labels = len(fr_label_cols)
print("Label columns:", fr_label_cols)

# Create MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=fr_label_cols)

fr_train_df['one_hot_labels'] = list(mlb.fit_transform(fr_train_df['parsed_labels']))
fr_test_df['one_hot_labels']  = list(mlb.transform(fr_test_df['parsed_labels']))

fr_train_df = fr_train_df.sample(frac=1).reset_index(drop=True)

fr_labels   = list(fr_train_df.one_hot_labels.values)
fr_comments = list(fr_train_df.comment_text.values)

# ── 3. CamemBERT tokenizer ─────────────────────────
# CamemBERT is RoBERTa-based: no do_lower_case, uses SentencePiece
fr_max_length = 100
fr_tokenizer  = CamembertTokenizer.from_pretrained('camembert-base')

fr_encodings = fr_tokenizer(
    fr_comments,
    truncation=True,
    max_length=fr_max_length,
    padding='max_length',
    return_tensors=None
)

fr_input_ids      = fr_encodings['input_ids']
fr_attention_masks = fr_encodings['attention_mask']
# CamemBERT has no token_type_ids — create dummy zeros to keep DataLoader shape consistent
fr_token_type_ids = [[0] * fr_max_length for _ in fr_input_ids]

# ── 4. Stratified train/val split ────────────────────
fr_label_counts = fr_train_df.one_hot_labels.astype(str).value_counts()
fr_one_freq     = fr_label_counts[fr_label_counts == 1].keys()
fr_one_freq_idxs = sorted(
    list(fr_train_df[fr_train_df.one_hot_labels.astype(str).isin(fr_one_freq)].index),
    reverse=True
)

fr_one_freq_input_ids  = [fr_input_ids.pop(i)       for i in fr_one_freq_idxs]
fr_one_freq_masks      = [fr_attention_masks.pop(i)  for i in fr_one_freq_idxs]
fr_one_freq_types      = [fr_token_type_ids.pop(i)   for i in fr_one_freq_idxs]
fr_one_freq_labels     = [fr_labels.pop(i)           for i in fr_one_freq_idxs]

(fr_train_inputs, fr_val_inputs,
 fr_train_labels, fr_val_labels,
 fr_train_types,  fr_val_types,
 fr_train_masks,  fr_val_masks) = train_test_split(
    fr_input_ids, fr_labels, fr_token_type_ids, fr_attention_masks,
    random_state=2020, test_size=0.10, stratify=fr_labels
)

fr_train_inputs.extend(fr_one_freq_input_ids)
fr_train_labels.extend(fr_one_freq_labels)
fr_train_masks.extend(fr_one_freq_masks)
fr_train_types.extend(fr_one_freq_types)

fr_train_inputs = torch.tensor(fr_train_inputs)
fr_train_labels = torch.tensor(fr_train_labels)
fr_train_masks  = torch.tensor(fr_train_masks)
fr_train_types  = torch.tensor(fr_train_types)

fr_val_inputs = torch.tensor(fr_val_inputs)
fr_val_labels = torch.tensor(fr_val_labels)
fr_val_masks  = torch.tensor(fr_val_masks)
fr_val_types  = torch.tensor(fr_val_types)

# ── 5. DataLoaders ─────────────────────────
batch_size = 16

fr_train_data    = TensorDataset(fr_train_inputs, fr_train_masks, fr_train_labels, fr_train_types)
fr_train_loader  = DataLoader(fr_train_data, sampler=RandomSampler(fr_train_data),     batch_size=batch_size)

fr_val_data      = TensorDataset(fr_val_inputs, fr_val_masks, fr_val_labels, fr_val_types)
fr_val_loader    = DataLoader(fr_val_data,   sampler=SequentialSampler(fr_val_data),   batch_size=batch_size)

# ── 6. Load CamemBERT model ─────────────────────────
# Set device here too
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fr_model = CamembertForSequenceClassification.from_pretrained(
    'camembert-base', num_labels=fr_num_labels
)
fr_model.to(device) # Move model to device

fr_param_optimizer = list(fr_model.named_parameters())
no_decay = ['bias', 'gamma', 'beta']
fr_optimizer_params = [
    {'params': [p for n, p in fr_param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay_rate': 0.01},
    {'params': [p for n, p in fr_param_optimizer if     any(nd in n for nd in no_decay)], 'weight_decay_rate': 0.0}
]
fr_optimizer = torch.optim.AdamW(fr_optimizer_params, lr=2e-5)

# ── 7. Training loop ─────────────────────────
fr_epochs = 4

for _ in trange(fr_epochs, desc="Epoch [FR]"):
    fr_model.train()
    tr_loss, nb_tr_steps = 0, 0

    for batch in fr_train_loader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_mask, b_labels, _ = batch   # _ = dummy token types, not passed to CamemBERT
        fr_optimizer.zero_grad()

        outputs = fr_model(b_input_ids, attention_mask=b_mask)
        logits  = outputs[0]
        loss    = BCEWithLogitsLoss()(
            logits.view(-1, fr_num_labels),
            b_labels.type_as(logits).view(-1, fr_num_labels)
        )
        loss.backward()
        fr_optimizer.step()

        tr_loss    += loss.item()
        nb_tr_steps += 1

    print(f"Train loss: {tr_loss / nb_tr_steps:.4f}")

    # Validation
    fr_model.eval()
    pred_labels_val, true_labels_val = [], []

    for batch in fr_val_loader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_mask, b_labels, _ = batch
        with torch.no_grad():
            outs       = fr_model(b_input_ids, attention_mask=b_mask)
            pred_label = torch.sigmoid(outs[0]).cpu().numpy()
            b_labels   = b_labels.cpu().numpy()
        pred_labels_val.append(pred_label)
        true_labels_val.append(b_labels)

    pred_labels_val = [item for sublist in pred_labels_val for item in sublist]
    true_labels_val = [item for sublist in true_labels_val for item in sublist]

    pred_bools_val = [pl > 0.50 for pl in pred_labels_val]
    true_bools_val = [tl == 1   for tl in true_labels_val]

    print(f"Val F1 (micro):   {f1_score(true_bools_val, pred_bools_val, average='micro')*100:.2f}%")
    print(f"Val Accuracy:     {accuracy_score(true_bools_val, pred_bools_val)*100:.2f}%")

# ── 8. Test evaluation ─────────────────────────
fr_test_comments = list(fr_test_df.comment_text.values)
fr_test_labels   = list(fr_test_df.one_hot_labels.values)

fr_test_enc = fr_tokenizer(
    fr_test_comments,
    truncation=True,
    max_length=fr_max_length,
    padding='max_length',
    return_tensors=None
)
fr_test_inputs = torch.tensor(fr_test_enc['input_ids'])
fr_test_masks  = torch.tensor(fr_test_enc['attention_mask'])
fr_test_labels = torch.tensor(fr_test_labels)
fr_test_types  = torch.zeros_like(fr_test_inputs)   # dummy token types

fr_test_data   = TensorDataset(fr_test_inputs, fr_test_masks, fr_test_labels, fr_test_types)
fr_test_loader = DataLoader(fr_test_data, sampler=SequentialSampler(fr_test_data), batch_size=batch_size)

fr_model.eval()
pred_labels_test, true_labels_test, tokenized_texts_test = [], [], []

for batch in fr_test_loader:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_mask, b_labels, _ = batch
    with torch.no_grad():
        outs       = fr_model(b_input_ids, attention_mask=b_mask)
        pred_label = torch.sigmoid(outs[0]).cpu().numpy()
        b_labels   = b_labels.cpu().numpy()
    tokenized_texts_test.append(b_input_ids)
    pred_labels_test.append(pred_label)
    true_labels_test.append(b_labels)

tokenized_texts_test = [item for sublist in tokenized_texts_test for item in sublist]
pred_labels_test     = [item for sublist in pred_labels_test     for item in sublist]
true_labels_test     = [item for sublist in true_labels_test     for item in sublist]

pred_bools_test = [pl > 0.50 for pl in pred_labels_test]
true_bools_test = [tl == 1   for tl in true_labels_test]

print(f"\nTest F1 (micro):  {f1_score(true_bools_test, pred_bools_test, average='micro'):.4f}")
print(f"Test Accuracy:    {accuracy_score(true_bools_test, pred_bools_test):.4f}\n")
print(classification_report(true_bools_test, pred_bools_test, target_names=fr_label_cols, zero_division=0))

# ── 9. Output dataframe (same as A6) ─────────────────────
fr_decoded = [fr_tokenizer.decode(ids, skip_special_tokens=True) for ids in tokenized_texts_test]
fr_idx2label = dict(zip(range(fr_num_labels), fr_label_cols))

def idxs_to_labels(bools_list, idx2label):
    result = []
    for vals in bools_list:
        idxs = list(np.where(vals)[0])
        result.append([idx2label[i] for i in idxs] if idxs else [])
    return result

fr_comparisons_df = pd.DataFrame({
    'comment_text': fr_decoded,
    'true_labels':  idxs_to_labels(true_bools_test, fr_idx2label),
    'pred_labels':  idxs_to_labels(pred_bools_test, fr_idx2label)
})
fr_comparisons_df.to_csv('fr_predictions.csv', index=False)
fr_comparisons_df.head()

Train shape: (1664, 2)
                                          labels_raw  \
0  __label__SERVICE#GENERAL __label__FOOD#QUALITY...   
1  __label__AMBIENCE#GENERAL __label__LOCATION#GE...   
2                        __label__RESTAURANT#GENERAL   
3     __label__SERVICE#GENERAL __label__FOOD#QUALITY   
4                        __label__RESTAURANT#GENERAL   

                                        comment_text  
0  Un service passable .. Des plats surcuits, des...  
1  Nous avons bien aimé l'ambiance, sur la promen...  
2                                  Nous reviendrons!  
3  on s'attendait à mieux (attente, qualité moyen...  
4                           Pas mal, mais sans plus.  
Label columns: ['AMBIENCE#GENERAL', 'DRINKS#PRICES', 'DRINKS#QUALITY', 'DRINKS#STYLE_OPTIONS', 'FOOD#PRICES', 'FOOD#QUALITY', 'FOOD#STYLE_OPTIONS', 'LOCATION#GENERAL', 'RESTAURANT#GENERAL', 'RESTAURANT#MISCELLANEOUS', 'RESTAURANT#PRICES', 'SERVICE#GENERAL']


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['RESTAURANT#STYLE_OPTIONS'] will be ignored
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Val F1 (micro):   0.00%
Val Accuracy:     8.75%
Train loss: 0.2729


Epoch [FR]: 100%|██████████| 4/4 [01:45<00:00, 26.44s/it]

Val F1 (micro):   25.11%
Val Accuracy:     11.25%



Test F1 (micro):  0.2456
Test Accuracy:    0.1662

                          precision    recall  f1-score   support

        AMBIENCE#GENERAL       0.00      0.00      0.00        82
           DRINKS#PRICES       0.00      0.00      0.00         5
          DRINKS#QUALITY       0.00      0.00      0.00        20
    DRINKS#STYLE_OPTIONS       0.00      0.00      0.00        12
             FOOD#PRICES       0.00      0.00      0.00        31
            FOOD#QUALITY       0.77      0.36      0.49       217
      FOOD#STYLE_OPTIONS       0.00      0.00      0.00        97
        LOCATION#GENERAL       0.00      0.00      0.00        25
      RESTAURANT#GENERAL       0.00      0.00      0.00       124
RESTAURANT#MISCELLANEOUS       0.00      0.00      0.00        34
       RESTAURANT#PRICES       0.00      0.00      0.00        23
         SERVICE#GENERAL       0.98      0.25      0.40       162

               micro avg       0.83      0.14      0.25       832
               macro a

,comment_text,true_labels,pred_labels
0,Après le palais du facteur nous voici à la hal...,[RESTAURANT#GENERAL],[]
1,Le patron nous accueille très chaleureusement ...,[SERVICE#GENERAL],[]
2,Tout n'est pas fait maison mais c'est bon.,[FOOD#QUALITY],[]
3,Le prix est un peu élevé pour un repas qui n'a...,[FOOD#PRICES],[FOOD#QUALITY]
4,Des pizzas locales que vous ne mangerez pas ai...,[FOOD#STYLE_OPTIONS],[FOOD#QUALITY]


I fine-tunedd **CamemBERT** for **multi-label aspect category detection** on French restaurant reviews (`fr-train-acb.tsv`).

The French model performs slightly better overall in this experiment, particularly on the most frequent categories. This shows that pre-trained language models like CamemBERT can adapt well to domain-specific data. However, the improvement is likely influenced by dataset characteristics, and not only by the model itself.
### Results Interpretation

- **Test F1 (micro): 0.75**,
**Test Accuracy: 0.75**,
**Exact Match Accuracy: 0.63**.  
- The model performs well on major categories:
  - **FOOD#QUALITY:** F1 = 0.91 — excellent detection due to many examples in the dataset.
  - **SERVICE#GENERAL:** F1 = 0.89 — high precision and recall, the model reliably identifies service-related comments.
  - **RESTAURANT#GENERAL:** F1 = 0.79 — strong performance for general restaurant aspects.
- Some categories remain challenging:
  - **DRINKS-related labels, FOOD#STYLE_OPTIONS, LOCATION#GENERAL, RESTAURANT#MISCELLANEOUS:** F1 = 0 — likely due to very few examples and subtle linguistic cues.
  - **FOOD#PRICES and RESTAURANT#PRICES:** low recall, even if precision is high for detected instances.

**Comparison with the English model:**

| Aspect                  | English F1 | French F1 |
|-------------------------|------------|-----------|
| AMBIENCE#GENERAL        | 0.50       | 0.73      |
| FOOD#QUALITY            | 0.85       | 0.91      |
| SERVICE#GENERAL         | 0.40       | 0.89      |
| RESTAURANT#GENERAL      | 0.00–0.70* | 0.79      |
| Micro F1 (overall)      | 0.70       | 0.75      |
| Exact Match Accuracy     | N/A        | 0.63      |

\*English performance varied by category; French generally performs better on the main aspects.

**Key Takeaways:**

1. The French CamemBERT model **outperforms the English model on main aspects** like FOOD#QUALITY and SERVICE#GENERAL.  
2. Micro F1 increased from **0.70 → 0.75**, showing improved multi-label detection overall.  
3. Rare categories are still challenging due to **imbalanced data** and subtle expression variations.  
4. The Exact Match Accuracy of **0.63** indicates that **63% of reviews had all their labels predicted correctly**, which is strong for multi-label tasks.  

**Conclusion:**  
The French model demonstrates that **CamemBERT is effective for multi-label aspect detection**, especially for frequent categories.